In [1]:
# STEP 1: Load the three CSV datasets.
import pandas as pd
import re

# Load the Kaggle and Olympedia datasets.

athletes = pd.read_csv("../data/athlete_events.csv")
bios = pd.read_csv("../data/bios.csv")
results = pd.read_csv("../data/results.csv", low_memory=False)

In [2]:
# Normalize names to account for characters lost in Kaggle.
# Replace Olympedia's bullet separator with a space.
# Remove non-ASCII characters, collapse whitespace, and use lowercase.

def normalize_name(name):
    if pd.isna(name):
        return ""

    name = str(name)
    name = name.replace("•", " ")
    name = name.encode("ascii", errors="ignore").decode("ascii")
    name = re.sub(r"\s+", " ", name)

    return name.strip().lower()

In [3]:
# Keep unique athlete IDs and names from Kaggle.
kaggle = athletes[["ID", "Name"]].drop_duplicates()
kaggle["normalized_name"] = kaggle["Name"].apply(normalize_name)

# Keep athlete IDs and names from Olympedia.
olympedia = bios[["athlete_id", "Used name"]].copy()
olympedia["normalized_name"] = olympedia["Used name"].apply(normalize_name)

In [4]:
# Get each athlete's participation years from Kaggle.
kaggle_years = athletes[["ID", "Year"]].drop_duplicates()

# Extract Olympic years from Olympedia results.
olympedia_years = results[["athlete_id", "Games"]].drop_duplicates()

olympedia_years["Year"] = pd.to_numeric(
    olympedia_years["Games"].str.extract(r"(\d{4})")[0],
    errors="coerce"
)

olympedia_years = olympedia_years[
    ["athlete_id", "Year"]
].drop_duplicates()

In [5]:
# First match athletes with identical normalized names.
matches = kaggle.merge(
    olympedia,
    on="normalized_name",
    how="inner"
)

# Attach the Olympic participation years from Kaggle.
matches = matches.merge(
    kaggle_years,
    on="ID"
)

# Keep only pairs with at least one overlapping Olympic year.
matches = matches.merge(
    olympedia_years,
    on=["athlete_id", "Year"]
)

# Each athlete pair only needs to appear once,
# even if they competed in several of the same years.
matches = matches.drop_duplicates(
    subset=["ID", "athlete_id"]
)

In [6]:
# Count how many Olympedia athletes match each Kaggle athlete.
candidate_counts = matches.groupby("ID")["athlete_id"].nunique()

# Show the distribution of candidate counts.
print(candidate_counts.value_counts().sort_index())

# Check how many Kaggle athletes have at least one match.
print("Total Kaggle athletes:", kaggle["ID"].nunique())
print("Matched Kaggle athletes:", matches["ID"].nunique())
print(
    "Unmatched Kaggle athletes:",
    kaggle["ID"].nunique() - matches["ID"].nunique()
)

athlete_id
1    67477
2      286
3       41
4        6
Name: count, dtype: int64
Total Kaggle athletes: 135571
Matched Kaggle athletes: 67810
Unmatched Kaggle athletes: 67761


In [7]:
# Select athletes with exactly one Olympedia candidate.
unique_ids = candidate_counts[candidate_counts == 1].index

unique_matches = matches[
    matches["ID"].isin(unique_ids)
].copy()

# Keep athletes with multiple candidates separately.
ambiguous_ids = candidate_counts[candidate_counts > 1].index

ambiguous_matches = matches[
    matches["ID"].isin(ambiguous_ids)
].copy()

print("Unique matches:", unique_matches["ID"].nunique())
print("Ambiguous athletes:", ambiguous_matches["ID"].nunique())

Unique matches: 67477
Ambiguous athletes: 333


In [8]:
# These Kaggle athletes have no candidate satisfying both
# the normalized-name and Olympic-year conditions.
unmatched = kaggle[
    ~kaggle["ID"].isin(matches["ID"])
]

print("Unmatched athletes:", len(unmatched))

# Inspect the first 20.
unmatched.head(20)

Unmatched athletes: 67761


,ID,Name,normalized_name
0,1,A Dijiang,a dijiang
2,3,Gunnar Nielsen Aaby,gunnar nielsen aaby
3,4,Edgar Lindenau Aabye,edgar lindenau aabye
4,5,Christine Jacoba Aaftink,christine jacoba aaftink
10,6,Per Knut Aaland,per knut aaland
26,8,"Cornelia ""Cor"" Aalten (-Strannood)","cornelia ""cor"" aalten (-strannood)"
28,9,Antti Sami Aalto,antti sami aalto
29,10,"Einar Ferdinand ""Einari"" Aalto","einar ferdinand ""einari"" aalto"
30,11,Jorma Ilmari Aalto,jorma ilmari aalto
31,12,Jyri Tapani Aalto,jyri tapani aalto


In [9]:
# Rename the Olympedia name column for clarity.
unique_matches = unique_matches.rename(
    columns={"Used name": "olympedia_name"}
)

# Replace Olympedia's bullet separators with spaces.
unique_matches["olympedia_name"] = (
    unique_matches["olympedia_name"]
    .str.replace("•", " ", regex=False)
)

# Find athletes whose original names differ.
different_names = unique_matches[
    unique_matches["Name"] != unique_matches["olympedia_name"]
]

different_names[
    ["ID", "Name", "olympedia_name"]
].head(20)

,ID,Name,olympedia_name
3,20,Kjetil Andr Aamodt,Kjetil André Aamodt
17,37,Ann Kristin Aarnes,Ann Kristin Aarønes
19,44,Meelis Aasme,Meelis Aasmäe
21,45,Hakon Aasns,Hakon Aasnæs
22,46,Hans Aasns,Hans Aasnæs
35,56,Ren Abadie,René Abadie
46,74,Mara Laura Abalo,María Laura Abalo
52,94,Ral Abatte,Raúl Abatte
53,99,Pter Abay,Péter Abay
54,100,Oszkr Abay-Nemes,Oszkár Abay-Nemes


In [10]:
# Look up a specific name.
unique_matches.loc[
    unique_matches["Name"] == "Jos Aceituno",
    ["Name", "olympedia_name"]
]

,Name,olympedia_name
407,Jos Aceituno,José Aceituno


In [11]:
# Start with all unique Kaggle athletes.
all_athletes = kaggle[["ID", "Name"]].copy()

# Add the Olympedia ID and corrected name where available.
all_athletes = all_athletes.merge(
    unique_matches[["ID", "athlete_id", "olympedia_name"]],
    on="ID",
    how="left"
)

# Initially, assume there was no match.
all_athletes["errormessage"] = "No name match found"

# Athletes with an Olympedia ID have a unique match.
all_athletes.loc[
    all_athletes["athlete_id"].notna(),
    "errormessage"
] = None

# Athletes with multiple candidates get a different error message.
all_athletes.loc[
    all_athletes["ID"].isin(ambiguous_ids),
    "errormessage"
] = "Multiple name matches found"

In [12]:
# Each Kaggle athlete should appear exactly once.
print("Original athletes:", kaggle["ID"].nunique())
print("Final athletes:", all_athletes["ID"].nunique())

# Check how many athletes have each matching status.
print(all_athletes["errormessage"].value_counts(dropna=False))

# Verify that no athlete IDs are duplicated.
assert all_athletes["ID"].is_unique

Original athletes: 135571
Final athletes: 135571
errormessage
No name match found            67761
None                           67477
Multiple name matches found      333
Name: count, dtype: int64


In [14]:
all_athletes.to_csv(
    "../data/olympedia_all_athletes.csv",
    index=False
)